In [1]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
from sqlalchemy import create_engine

# ==============================================================================
# 1. INITIALISATION : CONNEXION BRONZE
# ==============================================================================
DB_CONFIG = {
    'host': 'mariadb',
    'port': '3306',
    'user': 'root',
    'password': 'root',
    'database': 'data_quality_db'
}

# Connexion SQL
connection_str = f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
engine = create_engine(connection_str)

print("[INIT] Chargement des donnees depuis la table 'healthcare_bronze'...")
try:
    df = pd.read_sql("SELECT * FROM healthcare_bronze", con=engine)
    print(f"[INIT] Donnees chargees : {df.shape[0]} lignes.")
except Exception as e:
    print(f"[ERREUR] Impossible de lire la table Bronze : {e}")
    raise

df_clean = df.copy()



[INIT] Chargement des donnees depuis la table 'healthcare_bronze'...
[INIT] Donnees chargees : 55500 lignes.


In [2]:
# ==============================================================================
# 2. APPLICATION DES REGLES TECHNIQUES (DATA QUALITY)
# ==============================================================================
print("\n[PHASE TECHNIQUE] Nettoyage Structurel...")

# T01 : Completude (Suppression si cles manquantes)
init_len = len(df_clean)
df_clean = df_clean.dropna(subset=['Name', 'Date of Admission'])
print(f"   -> [T01] Lignes incompletes supprimees : {init_len - len(df_clean)}")

# T02 : Unicite (Dedoublonnage strict)
init_len = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Name', 'Date of Admission'], keep='first')
print(f"   -> [T02] Doublons techniques supprimes : {init_len - len(df_clean)}")

# T03 : Exactitude (Standardisation Title Case + Trim)
text_cols = ['Name', 'Doctor', 'Hospital', 'Medical Condition', 'Medication', 'Insurance Provider']
for col in text_cols:
    df_clean[col] = df_clean[col].str.title().str.strip()
print("   -> [T03] Textes standardises (Title Case).")

# T04 : Exactitude (Nettoyage Hopitaux)
df_clean['Hospital'] = df_clean['Hospital'].apply(lambda x: re.sub(r'(?i)\s+(LLC|Inc|Ltd|PLC|Group)$', '', str(x)))
print("   -> [T04] Noms d'hopitaux nettoyes.")

# T05 : Validite (Bornage Age)
mask_age = (df_clean['Age'] < 0) | (df_clean['Age'] > 120)
df_clean.loc[mask_age, 'Age'] = np.nan
print(f"   -> [T05] Ages aberrants mis a NaN : {mask_age.sum()}")

# T06 : Validite (Conformite Listes - Groupe Sanguin)
valid_blood = ['A+', 'A-', 'B+', 'B-', 'O+', 'O-', 'AB+', 'AB-']
mask_blood = ~df_clean['Blood Type'].isin(valid_blood)
df_clean.loc[mask_blood, 'Blood Type'] = 'Unknown'
print(f"   -> [T06] Groupes sanguins invalides corriges : {mask_blood.sum()}")



[PHASE TECHNIQUE] Nettoyage Structurel...
   -> [T01] Lignes incompletes supprimees : 0
   -> [T02] Doublons techniques supprimes : 5500
   -> [T03] Textes standardises (Title Case).
   -> [T04] Noms d'hopitaux nettoyes.
   -> [T05] Ages aberrants mis a NaN : 0
   -> [T06] Groupes sanguins invalides corriges : 0


In [3]:
# ==============================================================================
# 3. APPLICATION DES REGLES METIER (BUSINESS LOGIC)
# ==============================================================================
print("\n[PHASE METIER] Logique Fonctionnelle...")

# M01 : Finance (Facturation Positive)
neg_bills = (df_clean['Billing Amount'] < 0).sum()
df_clean['Billing Amount'] = df_clean['Billing Amount'].abs()
print(f"   -> [M01] Factures negatives redressees : {neg_bills}")

# Conversion Dates
df_clean['Date of Admission'] = pd.to_datetime(df_clean['Date of Admission'], errors='coerce')
df_clean['Discharge Date'] = pd.to_datetime(df_clean['Discharge Date'], errors='coerce')

# M02 : Actualite (Chronologie)
mask_chrono = df_clean['Discharge Date'] >= df_clean['Date of Admission']
invalid_dates = (~mask_chrono).sum()
df_clean = df_clean[mask_chrono]
print(f"   -> [M02] Incoherences chronologiques supprimees : {invalid_dates}")

# M03 : Actualite (Pas de dates futures)
mask_future = df_clean['Date of Admission'] > pd.Timestamp.now()
df_clean = df_clean[~mask_future]
print(f"   -> [M03] Admissions futures supprimees : {mask_future.sum()}")

# M04 : Coherence (Assurance vs Age)
mask_medicare = (df_clean['Age'] < 65) & (df_clean['Insurance Provider'] == 'Medicare')
df_clean.loc[mask_medicare, 'Insurance Provider'] = 'Blue Cross'
print(f"   -> [M04] Correction Medicare (<65 ans) : {mask_medicare.sum()}")

# M07 : Coherence (Facture vs Sejour)
df_clean['Length_of_Stay'] = (df_clean['Discharge Date'] - df_clean['Date of Admission']).dt.days
mask_bill_suspicious = (
    ((df_clean['Length_of_Stay'] <= 1) & (df_clean['Billing Amount'] > 20000)) | 
    ((df_clean['Length_of_Stay'] > 30) & (df_clean['Billing Amount'] < 500))
)
df_clean['Alert_Billing_Anomaly'] = mask_bill_suspicious
print(f"   -> [M07] Anomalies Facture/Sejour flaggees : {mask_bill_suspicious.sum()}")

# M05 & M06 : Flagging Clinique (Medicaments & Tests)
# M06 Tests
mask_test = (df_clean['Medical Condition'] == 'Diabetes') & (df_clean['Test Results'] == 'Normal')
df_clean['Alert_Test'] = mask_test

# M05 Medicaments
incompatible_meds = {
    'Asthma': ['Penicillin', 'Lipitor'],
    'Cancer': ['Aspirin', 'Ibuprofen'],
    'Obesity': ['Penicillin', 'Aspirin']
}
correction_map = {'Asthma': 'Albuterol', 'Cancer': 'Chemo', 'Obesity': 'Diet'}

df_clean['Alert_Medication'] = False
df_clean['Medication_Corrected'] = df_clean['Medication']

count_meds = 0
for cond, meds in incompatible_meds.items():
    mask = (df_clean['Medical Condition'] == cond) & (df_clean['Medication'].isin(meds))
    if mask.sum() > 0:
        df_clean.loc[mask, 'Alert_Medication'] = True
        df_clean.loc[mask, 'Medication_Corrected'] = correction_map.get(cond, 'Other')
        count_meds += mask.sum()

print(f"   -> [M05/M06] Incoherences Cliniques flaggees : {count_meds + mask_test.sum()}")




[PHASE METIER] Logique Fonctionnelle...
   -> [M01] Factures negatives redressees : 96
   -> [M02] Incoherences chronologiques supprimees : 0
   -> [M03] Admissions futures supprimees : 0
   -> [M04] Correction Medicare (<65 ans) : 6983
   -> [M07] Anomalies Facture/Sejour flaggees : 982
   -> [M05/M06] Incoherences Cliniques flaggees : 12787


In [4]:
# ==============================================================================
# 4. EXPORT FINAL (COUCHE SILVER)
# ==============================================================================
print("\n[EXPORT] Sauvegarde vers la couche Silver...")
try:
    df_clean.to_sql('healthcare_silver', con=engine, if_exists='replace', index=False)
    print(f"[SUCCES] Table 'healthcare_silver' creee avec succes.")
    print(f"         Volume Final : {df_clean.shape[0]} lignes.")
    print("         Le dataset est pret pour le Dashboard.")
except Exception as e:
    print(f"[ERREUR] Echec de l'export SQL : {e}")


[EXPORT] Sauvegarde vers la couche Silver...
[SUCCES] Table 'healthcare_silver' creee avec succes.
         Volume Final : 50000 lignes.
         Le dataset est pret pour le Dashboard.
